In [47]:
import re
import textwrap

import aiohttp
from bs4 import BeautifulSoup
from google import genai
from pydantic import BaseModel, Field


In [6]:
link = "https://www.lizapanelim.com/%d7%a9%d7%aa%d7%99-%d7%a2%d7%95%d7%92%d7%95%d7%aa-%d7%a9%d7%99%d7%a9-%d7%92%d7%91%d7%99%d7%a0%d7%94-%d7%a0%d7%94%d7%93%d7%a8%d7%95%d7%aa/"
async with aiohttp.ClientSession() as session:
    async with session.get(link) as response:
        print("Status:", response.status)
        html = await response.text()
        # print("Body:", html, "...")


Status: 200


In [ ]:
soup = BeautifulSoup(html, "html.parser")
text = soup.get_text()
text = textwrap.dedent(text)
text = re.sub(r"\n+", "\n", text)
# print(text)

In [48]:
class Recipe(BaseModel):
    ingredients: list[str]
    instructions: list[str]
    title: str = Field(..., description="short one liner title of the dish")
    description: str = Field(..., description="short description of the dish")


client = genai.Client(vertexai=True, project="personal-ai-264306", location="us-central1")
response = client.models.generate_content(
    model="gemini-2.5-flash-preview-05-20",
    contents=(
        "Given the following text from a web page, extract the exact text of the ingredients and instructions for the recipe. Use the same language as in the text."
        "Write the ingredients and instructions in the same order and exact values as they are in the text."
        "<text>" + text + "</text>"
    ),
    config={
        "response_mime_type": "application/json",
        "response_schema": Recipe,
    },
)

In [49]:
response

GenerateContentResponse(candidates=[Candidate(content=Content(parts=[Part(video_metadata=None, thought=None, inline_data=None, file_data=None, code_execution_result=None, executable_code=None, function_call=None, function_response=None, text='{\n  "ingredients": [\n    "4 ביצים",\n    "⅔ כוס שמן צמחי (160 מ”ל)",\n    "⅔ כוס שמנת מתוקה (160 מ”ל)",\n    "⅔ כוס סוכר (כ-130 גרם)",\n    "2 וחצי כוסות קמח תופח או קמח עם כפית א. אפייה",\n    "כפית שמרים יבשים",\n    "200 גרם גבינת שמנת (אני השתמשתי בארלה)",\n    "⅓ כוס סוכר כ-70 גרם",\n    "שקית סוכר וניל",\n    "חצי כפית גרד לימון",\n    "כוס פטל טרי /מישמשים חתוכים/ דובדבנים או שזיפים או נקטרינות",\n    "2-3 כפות ממרח שוקולד עם 1-2 כפות מים רותחים לדילול",\n    "כף ריבת מישמש עם כף מים רותחים לזיגוג מבריק מעל (לא חובה)"\n  ],\n  "instructions": [\n    "נדליק תנור ל-160 מעלות חום עליון תחתון",\n    "נטרוף במרץ: ביצים, שמן, שמנת מתוקה סוכר, נוסיף קמח, אבקת אפייה נערבב היטב, נוסיף כפית שמרים יבשים, נערבב עד איחוד. נניח בצד ל-10 דק שהשמרים “יעב

In [52]:
response.parsed.description

'בלילה אחת ושתי עוגות גבינה שונות, טעימות ומיוחדות'